In [1]:
# importing dependecies
import random                          # Used to randomly pick one caption per image
from torch.utils.data import Dataset, DataLoader   # Tools for handling datasets and batching
from torchvision import transforms     # Image preprocessing utilities
from datasets import load_dataset      # To load dataset from Hugging Face
from collections import Counter        # Helps count word frequencies
import torch

In [2]:
# Loading the dataset
dataset = load_dataset("jxie/flickr8k")
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…):   0%|          | 0.00/414M [00:00<?, ?B/s]

data/train-00001-of-00002-2173151d8cd6c7(…):   0%|          | 0.00/414M [00:00<?, ?B/s]

data/validation-00000-of-00001-7025a2b59(…):   0%|          | 0.00/138M [00:00<?, ?B/s]

data/test-00000-of-00001-42a2661d12c73e4(…):   0%|          | 0.00/137M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [3]:
train_data.features

{'image': Image(mode=None, decode=True),
 'caption_0': Value('string'),
 'caption_1': Value('string'),
 'caption_2': Value('string'),
 'caption_3': Value('string'),
 'caption_4': Value('string')}

In [4]:
# image transform resizing and normalizing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])])

In [5]:
def build_vocab(dataset, freq_threshold=4):
    """
    Builds a vocabulary (word -> index mapping)
    Parameters:
    - dataset: training dataset
    - freq_threshold: minimum frequency for a word to be included
    Returns:
    - word2idx: dictionary mapping word -> integer
    - idx2word: reverse mapping
    """

    counter = Counter()   # counts frequency of each word
    # Loop over all samples in dataset
    for sample in dataset:
        captions = [
           sample["caption_0"],
           sample["caption_1"],
           sample["caption_2"],
           sample["caption_3"],
           sample["caption_4"]]
        for caption in captions:
            tokens = caption.lower().split()
            counter.update(tokens)
    # Special tokens
    word2idx = {
        "<pad>": 0,    # used for padding shorter sequences
        "<start>": 1,  # start of sentence
        "<end>": 2,    # end of sentence
        "<unk>": 3  }   # unknown words
    idx = 4   # next index after special tokens

    # Add words that appear more than freq_threshold
    for word, freq in counter.items():
        if freq >= freq_threshold:
            word2idx[word] = idx
            idx += 1
    # Reverse mapping (index -> word)
    idx2word = {v: k for k, v in word2idx.items()}
    return word2idx, idx2word

In [6]:
# Build vocabulary only from training data
word2idx, idx2word = build_vocab(train_data)

In [7]:
def process_caption(caption, word2idx, max_len=20):
    """
    Converts a caption into:
    - input sequence
    - target sequence
    Steps:
    1. Tokenize
    2. Add <start> and <end>
    3. Convert words -> indices
    4. Pad to fixed length
    5. Create input-target pairs
    """
    tokens = caption.lower().split()   # Convert sentence to lowercase and split into words
    tokens = ["<start>"] + tokens + ["<end>"]  # Add special tokens
    # Convert words to indices and in case not found add <unk> special token
    sequence = [word2idx.get(word, word2idx["<unk>"]) for word in tokens]
    # Padding / truncation
    if len(sequence) < max_len:
        # Add <pad> tokens to reach max_len
        sequence += [word2idx["<pad>"]] * (max_len - len(sequence))
    else: sequence = sequence[:max_len] # truncate if the sequence is too long

    # Create input and target sequences
    input_seq  = sequence[:-1]
    target_seq = sequence[1:]
    return torch.tensor(input_seq), torch.tensor(target_seq)

In [8]:
class FlickrDataset(Dataset):
    """
    This class defines:
    - how many samples we have (__len__)
    - how to get one sample (__getitem__)
    """

    def __init__(self, data, word2idx, transform, max_len=20):
        """
        Parameters:
        - data: dataset split (train/val/test)
        - word2idx: vocabulary dictionary
        - transform: image preprocessing
        - max_len: max caption length
        """
        self.data = data
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        """Returns total number of samples"""
        return len(self.data)

    def __getitem__(self, idx):
        """
        Returns ONE sample:
        - image tensor
        - input sequence
        - target sequence
        """
        sample = self.data[idx]
        image = sample["image"]          # PIL image
        captions = [                     # list of 5 captions
           sample["caption_0"],
           sample["caption_1"],
           sample["caption_2"],
           sample["caption_3"],
           sample["caption_4"]]
        caption = random.choice(captions) #randomly choose one

        # Apply preprocessing to image
        image = self.transform(image)
        # Convert caption to sequences
        input_seq, target_seq = process_caption(
            caption, self.word2idx, self.max_len)
        return image, input_seq, target_seq

In [9]:
# Create dataset instances for each split
train_dataset = FlickrDataset(train_data, word2idx, transform)
val_dataset   = FlickrDataset(val_data, word2idx, transform)
test_dataset  = FlickrDataset(test_data, word2idx, transform)

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [11]:
# Take one batch and print shapes
for images, inputs, targets in train_loader:
    print("Images shape :", images.shape)    # (batch_size, 3, 224, 224)
    print("Inputs shape :", inputs.shape)    # (batch_size, max_len-1)
    print("Targets shape:", targets.shape)   # (batch_size, max_len-1)
    break

Images shape : torch.Size([32, 3, 224, 224])
Inputs shape : torch.Size([32, 19])
Targets shape: torch.Size([32, 19])
